# AUP-ZU3 — DMA Loopback Through DDR (PYNQ)

This notebook exercises the **AXI DMA loopback** design built from `spec/hardware_spec.md`.

**Data path:**

```
CPU  ──write──▶  DDR (source buffer)
                    │  DMA MM2S reads
                    ▼
              axis_filter  (32-bit AXI-Stream passthrough)
                    │  DMA S2MM writes
                    ▼
                 DDR (destination buffer)  ──read──▶  CPU verifies
```

Because `axis_filter` is a wire-through passthrough, the destination buffer must end up **bit-for-bit identical** to the source buffer.

**Design facts (from the hardware spec):**

| Item | Value |
|------|-------|
| Board | AUP-ZU3 8GB (`xczu3eg-sfvc784-2-e`) |
| AXI DMA control base | `0xA000_0000` (64 KB) |
| Stream width | 32-bit (`tdata[31:0]`, `tkeep[3:0]`) |
| Scatter-Gather | Disabled (simple mode) |
| DMA buffers | PS DDR (allocated contiguously via PYNQ) |
| PL clock | 100 MHz |


## 1. Prerequisites

Run this notebook **on the AUP-ZU3 board** in a PYNQ image (Python 3, `pynq` installed).

Copy these two files into the **same directory as this notebook** (they must share the same base name):

- `design_1_wrapper.bit`  — the bitstream
- `design_1_wrapper.hwh`  — the hardware hand-off metadata

Both are staged for you in this `pynq/` folder. If you rebuild the design, refresh them with:

```bash
cp vivado/aup_zu3_dma.runs/impl_1/design_1_wrapper.bit                     pynq/design_1_wrapper.bit
cp vivado/aup_zu3_dma.gen/sources_1/bd/design_1/hw_handoff/design_1.hwh    pynq/design_1_wrapper.hwh
```


## 2. Load the overlay

`Overlay(...)` downloads the bitstream to the PL and parses the `.hwh` so every IP becomes a Python attribute.

In [ ]:
from pynq import Overlay, allocate
import numpy as np

overlay = Overlay("design_1_wrapper.bit")

# List the IP blocks discovered in the design
overlay?  # opens docstring with the IP map; or use the line below
print(overlay.ip_dict.keys())

## 3. Grab a handle to the AXI DMA

PYNQ wraps `axi_dma` with a driver that exposes two channels:

- `sendchannel`  → **MM2S** (DDR → stream, feeds `axis_filter`)
- `recvchannel`  → **S2MM** (stream → DDR, drains `axis_filter`)

In [ ]:
dma = overlay.axi_dma_0
mm2s = dma.sendchannel
s2mm = dma.recvchannel
print("AXI DMA ready:", dma)

## 4. Allocate DDR buffers

`allocate()` returns a NumPy-like array backed by **physically contiguous DDR** with a known physical address the DMA can reach. We use `uint32` to match the 32-bit stream width.

> The DMA sees these buffers through `S_AXI_HP0_FPD`, which is mapped to DDR at `0x0000_0000` (2 GB) in the design's address map.

In [ ]:
N = 256  # number of 32-bit words to loop back

in_buf  = allocate(shape=(N,), dtype=np.uint32)
out_buf = allocate(shape=(N,), dtype=np.uint32)

print(f"in_buf  phys addr = 0x{in_buf.physical_address:08X}")
print(f"out_buf phys addr = 0x{out_buf.physical_address:08X}")

## 5. Fill the source buffer with test data

A mix of a known pattern and pseudo-random data makes mismatches easy to spot. `out_buf` is pre-cleared so a partial/failed transfer is obvious.

In [ ]:
rng = np.random.default_rng(0xC0FFEE)
in_buf[:4] = [0xDEADBEEF, 0x12345678, 0xCAFEBABE, 0xA5A5A5A5]
in_buf[4:] = rng.integers(0, 0xFFFFFFFF, size=N - 4, dtype=np.uint32)
out_buf[:] = 0

print("First 4 source words:", [hex(x) for x in in_buf[:4]])

## 6. Run the transfer

Start **both** channels, then wait for each to finish. In simple mode each `transfer()` moves one buffer; `wait()` blocks until the channel's `IOC` (transfer-complete) flag is set.

Order matters: arm the receive (S2MM) side before/together with the send (MM2S) side so no stream beats are dropped.

In [ ]:
import time

t0 = time.perf_counter()
s2mm.transfer(out_buf)   # arm receive first
mm2s.transfer(in_buf)    # then start sending
mm2s.wait()
s2mm.wait()
t1 = time.perf_counter()

bytes_moved = N * 4
print(f"Transfer done: {bytes_moved} bytes in {(t1 - t0) * 1e6:.1f} us")

## 7. Verify the loopback

Since `axis_filter` is a passthrough, `out_buf` must equal `in_buf` exactly.

In [ ]:
match = np.array_equal(in_buf, out_buf)
print("First 4 result words:", [hex(int(x)) for x in out_buf[:4]])

if match:
    print(f"\n✅ PASS — all {N} words match. DMA loopback through DDR + axis_filter verified.")
else:
    n_bad = int(np.count_nonzero(in_buf != out_buf))
    first = int(np.argmax(in_buf != out_buf))
    print(f"\n❌ FAIL — {n_bad} mismatched words. First at index {first}: "
          f"sent 0x{int(in_buf[first]):08X}, got 0x{int(out_buf[first]):08X}")

## 8. Clean up

Free the contiguous DDR buffers so the memory returns to the allocator.

In [ ]:
in_buf.freebuffer()
out_buf.freebuffer()
print("Buffers freed.")

## Troubleshooting

| Symptom | Likely cause / fix |
|---------|--------------------|
| `mm2s.wait()` / `s2mm.wait()` hangs forever | Stream stalled. Check the MM2S→`axis_filter`→S2MM connections and that `aresetn` is de-asserted. Re-download the overlay to reset the DMA. |
| `KeyError: 'axi_dma_0'` | The `.hwh` doesn't match the loaded `.bit`, or the DMA instance has a different name. Check `overlay.ip_dict.keys()`. |
| Output is all zeros | Receive channel not armed, or `out_buf` too small. Ensure `s2mm.transfer(out_buf)` ran and `len(out_buf) >= len(in_buf)`. |
| Partial/garbled data | Buffer dtype mismatch. Keep `uint32` to match the 32-bit stream. |
| `RuntimeError: DMA channel not idle` | A previous transfer didn't complete. Re-run the overlay load cell to reset. |
